# Reconnaissance d'expressions faciales (FER2013)

Projet Deep Learning - Duc Duy et Simon

But : reconnaître 7 expressions (angry, disgust, fear, happy, neutral, sad, surprise) sur des visages 48x48 en niveaux de gris.

Dataset : FER2013, Kaggle `msambare/fer2013`

Plan :
1. données
2. baseline dense
3. CNN
4. entraînement
5. évaluation et erreurs
6. expériences

et une démo à la fin.

### Étape 0 - config

À lancer en premier. On définit les constantes utilisées partout (dossier des données, classes, taille des images, seed) et on télécharge FER2013 avec kagglehub si `data/fer2013` n'existe pas encore. Ensuite on supprime les images vides : FER2013 contient quelques images unies (toutes noires ou grises, sans visage) qui n'apprennent que du bruit au modèle.

La seed (42) sert à avoir les mêmes résultats à chaque exécution, sinon on ne pourrait pas comparer les expériences de la partie 6.

In [ ]:
# Etape 0 : config + téléchargement des données
import os
import shutil
from pathlib import Path

import numpy as np
from PIL import Image
from tensorflow import keras

# on se met à la racine du repo (dossier avec le .gitignore), sinon on reste là (cas Colab)
racine = Path.cwd()
while racine != racine.parent and not (racine / ".gitignore").exists():
    racine = racine.parent
if not (racine / ".gitignore").exists():
    racine = Path.cwd()
os.chdir(racine)
print("racine:", racine)

DATA_DIR = Path("data/fer2013")
CLASS_NAMES = ["angry", "disgust", "fear", "happy", "neutral", "sad", "surprise"]  # l'ordre = position du 1 dans le one-hot
IMAGE_SIZE = (48, 48)
SEED = 42

keras.utils.set_random_seed(SEED)

if (DATA_DIR / "train").is_dir() and (DATA_DIR / "test").is_dir():
    print("données déjà là")
else:
    import kagglehub
    print("téléchargement de FER2013...")
    source = Path(kagglehub.dataset_download("msambare/fer2013"))
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    for split in ["train", "test"]:
        if not (DATA_DIR / split).exists():
            shutil.copytree(source / split, DATA_DIR / split)
    print("copié dans", DATA_DIR)

# on enlève les images vides (presque tous les pixels pareils = pas de visage)
retirees = []
for chemin in sorted(DATA_DIR.rglob("*.jpg")):
    with Image.open(chemin) as img:
        pixels = np.array(img)
    if pixels.std() < 1:
        retirees.append(chemin)
for chemin in retirees:
    chemin.unlink()
print("images vides retirées:", len(retirees))
for chemin in retirees:
    print(chemin)

## Partie 1 - Données

### Le dataset : FER2013

| | |
|---|---|
| Source | Kaggle, [msambare/fer2013](https://www.kaggle.com/datasets/msambare/fer2013) (version en images, un dossier par classe) |
| Auteurs | Pierre-Luc Carrier et Aaron Courville, pour le challenge ICML 2013 (Goodfellow et al., 2013, [arXiv:1307.0414](https://arxiv.org/abs/1307.0414)) |
| Licence | DbCL v1.0 (indiquée sur Kaggle), on l'utilise pour un projet de cours |
| Collecte | images trouvées sur Google Images avec des mots-clés d'émotions, visages recadrés automatiquement puis vérifiés à la main, réduits en 48x48 gris |
| Nombre d'images | 35 887 (28 709 en train, 7 178 en test) |
| Images vides | 13 images sans visage (12 en train, 1 en test), qu'on supprime à l'étape 0 |
| Classes | 7 : angry, disgust, fear, happy, neutral, sad, surprise |
| Taille | 48 x 48 pixels |
| Format | niveaux de gris (1 canal), fichiers .jpg |
| Déséquilibre | oui : 436 images disgust contre 7 215 happy dans train |

On a pris FER2013 parce qu'il est public, très utilisé, et que les visages sont déjà découpés. C'est ce qu'il faut pour les parties 1 à 6 où on part d'un visage déjà extrait.

Les chiffres sont vérifiés dans 1a et 1b.

### 1a - Nombre d'images par classe

Chaque dossier de classe = une étiquette. On compte les images et on fait un graphique pour voir le déséquilibre (c'est pour ça qu'en partie 5 on regarde le rappel par classe et pas juste l'accuracy).

In [ ]:
import matplotlib.pyplot as plt

nombres_train = []  # nb d'images train par classe, dans l'ordre de CLASS_NAMES

for split in ["train", "test"]:
    total = 0
    for classe in CLASS_NAMES:
        n = len(list((DATA_DIR / split / classe).glob("*.jpg")))
        total += n
        print(split, classe, n)
        if split == "train":
            nombres_train.append(n)
    print(split, "total:", total)
    print()

plt.figure(figsize=(8, 4))
plt.barh(CLASS_NAMES, nombres_train)
plt.gca().invert_yaxis()
plt.xlabel("nombre d'images (train)")
plt.title("images par classe (train)")
plt.show()

print("ratio max/min:", round(max(nombres_train) / min(nombres_train), 1))

### 1b - Exemples et vérification du format

On affiche 5 images par classe pour voir à quoi ça ressemble. Ensuite on vérifie sur toutes les images (train + test) que c'est bien du 48x48, en gris (mode "L" dans PIL) et en .jpg. Si une image avait une autre taille le Flatten planterait.

In [ ]:
from collections import Counter

N_EXEMPLES = 5

fig, axes = plt.subplots(len(CLASS_NAMES), N_EXEMPLES, figsize=(10, 14))
for ligne, classe in enumerate(CLASS_NAMES):
    fichiers = sorted((DATA_DIR / "train" / classe).glob("*.jpg"))[:N_EXEMPLES]
    for col, chemin in enumerate(fichiers):
        with Image.open(chemin) as img:
            axes[ligne, col].imshow(img, cmap="gray")
        axes[ligne, col].set_xticks([])
        axes[ligne, col].set_yticks([])
    axes[ligne, 0].set_ylabel(classe)
plt.suptitle("5 exemples par classe")
plt.tight_layout()
plt.show()
# extension = format des fichiers
# mode = type de l'image (ex: "L" pour grayscale)
# probl
# vérif du format sur toutes les images
tailles = Counter() # counter c'est pour compter combien d'images ont chaque taille
modes = Counter() # counter c'est pour compter combien d'images ont chaque mode
extensions = Counter() # counter c'est pour compter combien d'images ont chaque extension
modes = Counter() # counter c'est pour compter combien d'images ont chaque mode
problemes = []
n_images = 0
for split in ["train", "test"]:  # lancer 2 fois la boucle, 1 pour l'entraînement et 1 pour le test
    for classe in CLASS_NAMES:
        for chemin in sorted((DATA_DIR / split / classe).iterdir()): # chemin du dossier data/<split>/<classe> genre data/fer2013/train/angry et parcours ses éléments
            n_images += 1
            try:
                with Image.open(chemin) as img:
                    taille = img.size
                    mode = img.mode
            except OSError:
                problemes.append(chemin)
                continue
            ext = chemin.suffix.lower()
            tailles[taille] += 1
            modes[mode] += 1
            extensions[ext] += 1
            if taille != IMAGE_SIZE or mode != "L" or ext != ".jpg":
                problemes.append(chemin)

print("images vérifiées:", n_images)
print("tailles:", dict(tailles))
print("modes:", dict(modes))
print("extensions:", dict(extensions))
print("problèmes:", len(problemes))
for p in problemes[:20]:
    print(p)

### 1c - Préparation : train / validation / test

On crée les datasets avec `image_dataset_from_directory` :
- pas de redimensionnement, les images font déjà 48x48 (vu en 1b), donc une image = (48, 48, 1)
- normalisation : on divise les pixels par 255 pour passer de [0, 255] à [0, 1], l'entraînement est plus stable avec des petites valeurs
- labels : le nom du dossier donne la classe, encodée en one-hot avec `label_mode="categorical"` (happy = [0,0,0,1,0,0,0]). C'est ce qu'il faut pour softmax + categorical_crossentropy
- validation : 20% du dossier train (5 739 images). Elle sert à suivre l'entraînement et à choisir les modèles, le modèle n'apprend jamais dessus
- test : le dossier test (7 177 images). On n'y touche pas avant la toute fin (partie 6), sinon le score final serait faussé

`cache()` seulement sur val et test, sur train ça bloquerait l'ordre des images alors qu'on veut qu'il change à chaque époque.

In [ ]:
import tensorflow as tf

BATCH_SIZE = 64
VALIDATION_SPLIT = 0.2

train_ds, val_ds = keras.utils.image_dataset_from_directory(
    DATA_DIR / "train",
    color_mode="grayscale",
    image_size=IMAGE_SIZE,
    label_mode="categorical",
    class_names=CLASS_NAMES,
    batch_size=BATCH_SIZE,
    validation_split=VALIDATION_SPLIT,
    subset="both",
    seed=SEED,  # même découpage train/val à chaque fois
)

test_ds = keras.utils.image_dataset_from_directory(
    DATA_DIR / "test",
    color_mode="grayscale",
    image_size=IMAGE_SIZE,
    label_mode="categorical",
    class_names=CLASS_NAMES,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

print("train:", len(train_ds.file_paths))
print("val:", len(val_ds.file_paths))
print("test:", len(test_ds.file_paths))


def normaliser(images, labels):
    return images / 255.0, labels


train_ds = train_ds.map(normaliser).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.map(normaliser).cache().prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.map(normaliser).cache().prefetch(tf.data.AUTOTUNE)
# map est utilisé pour appliquer la fonction de normalisation à chaque lot d'images
# prefetch permet de préparer les lots suivants pendant que le modèle s'entraîne sur le lot actuel, ce qui améliore les performances.
# cache permet de stocker les données en mémoire après la première lecture, ce qui accélère les accès ultérieurs.

# petit check sur un lot
images_batch, labels_batch = next(iter(train_ds))
print(images_batch.shape, labels_batch.shape)
print("pixels min/max:", float(images_batch.numpy().min()), float(images_batch.numpy().max()))

## Partie 2 - Baseline dense

Le modèle le plus simple possible, juste pour avoir un point de comparaison avec le CNN. On suit le schéma du sujet : Flatten, Dense 128 (ReLU), Dense 7 (softmax).

Flatten met les 48x48 = 2304 pixels dans un seul vecteur. Chaque neurone de la couche cachée fait une somme pondérée de ces pixels plus un biais, puis ReLU. En sortie softmax donne 7 probabilités qui font 1 au total, la classe prédite est la plus probable.

Le problème attendu : après Flatten le réseau ne sait plus quels pixels sont voisins, un visage un peu décalé devient un vecteur complètement différent. C'est ce que le CNN doit corriger.

Compilation : categorical_crossentropy (labels one-hot + softmax), Adam, et l'accuracy comme métrique.

In [ ]:
from tensorflow.keras import layers

NUM_CLASSES = len(CLASS_NAMES)

baseline = keras.Sequential(
    [
        keras.Input(shape=IMAGE_SIZE + (1,)),  # (48, 48, 1)
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dense(NUM_CLASSES, activation="softmax"),
    ],
    name="baseline_dense",
)

baseline.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
baseline.summary()

Comment il apprend, en gros : pour chaque lot de 64 images on fait la propagation avant (l'image traverse les couches jusqu'aux 7 probas), on calcule la loss, ici -log(proba donnée à la vraie classe), puis la rétropropagation calcule le gradient de la loss pour chaque poids. La descente de gradient (Adam, lr 0.001) bouge ensuite chaque poids dans le sens qui fait baisser la loss.

Il y a 295 943 paramètres à apprendre : 2304 x 128 + 128 pour la couche cachée, 128 x 7 + 7 pour la sortie.

### 2b - Entraînement de la baseline

20 époques fixes, sans early stopping ici pour bien voir le surapprentissage sur les courbes. val_loss et val_accuracy sont calculées sur la validation, sur laquelle le modèle n'apprend pas.

In [ ]:
EPOCHS_BASELINE = 20

historique_baseline = baseline.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_BASELINE)
# fit est utilisé pour entraîner le modèle sur les données d'entraînement et évaluer sa performance sur les données de validation à chaque époque.

### 2c - Courbes et score

Pour savoir si le score est bon on le compare au hasard (1/7, environ 14%) et à un modèle qui répondrait toujours happy, la classe la plus fréquente (environ 25%).

In [ ]:
import numpy as np

h = historique_baseline.history
epoques = range(1, len(h["loss"]) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(epoques, h["loss"], label="train")
ax1.plot(epoques, h["val_loss"], label="val")
ax1.set_title("loss")
ax1.set_xlabel("époque")
ax1.legend()
ax2.plot(epoques, h["accuracy"], label="train")
ax2.plot(epoques, h["val_accuracy"], label="val")
ax2.set_title("accuracy")
ax2.set_xlabel("époque")
ax2.legend()
plt.suptitle("baseline dense")
plt.show()

perte_val, accuracy_val = baseline.evaluate(val_ds, verbose=0)

# nb d'images par classe dans la val, pour le repère "toujours la même classe"
comptes = np.zeros(NUM_CLASSES)
for images, labels in val_ds:
    comptes += labels.numpy().sum(axis=0)

print("accuracy val baseline:", round(accuracy_val * 100, 1), "%")
print("hasard:", round(100 / NUM_CLASSES, 1), "%")
print("toujours", CLASS_NAMES[int(comptes.argmax())], ":", round(comptes.max() / comptes.sum() * 100, 1), "%")

### 2d - Sortie pour une image

Les 7 probabilités données par la baseline pour une image de validation.

In [ ]:
images_val, labels_val = next(iter(val_ds))

probabilites = baseline.predict(images_val[:1], verbose=0)[0] # prédictions pour la première image du lot de validation
vraie_classe = CLASS_NAMES[int(labels_val[0].numpy().argmax())]
classe_predite = CLASS_NAMES[int(probabilites.argmax())]

plt.figure(figsize=(2.5, 2.5))
plt.imshow(images_val[0], cmap="gray", vmin=0, vmax=1)
plt.title("vraie classe: " + vraie_classe)
plt.axis("off")
plt.show()

for classe, p in zip(CLASS_NAMES, probabilites):
    print(classe, round(float(p), 2))
print("somme:", round(float(probabilites.sum()), 2))
print("prédit:", classe_predite, "/ vrai:", vraie_classe)

## Partie 3 - CNN

On part du schéma du sujet : Conv2D + ReLU, MaxPooling, Conv2D + ReLU, MaxPooling, Flatten, Dense, softmax.

Les filtres 3x3 glissent sur l'image et détectent des petits motifs (bords, coins) n'importe où dans l'image, ce que la baseline ne savait pas faire. Le MaxPooling garde le max de chaque carré 2x2 : la taille est divisée par 2 et le modèle est moins sensible à un petit décalage. Le 2e bloc combine les motifs simples en motifs plus grands (un oeil, une bouche).

Même compilation que la baseline pour pouvoir comparer.

In [ ]:
# on remet la seed avant de créer le modèle, pour partir des mêmes poids que les expériences de la partie 6
keras.utils.set_random_seed(SEED)

cnn = keras.Sequential(
    [
        keras.Input(shape=IMAGE_SIZE + (1,)),
        layers.Conv2D(32, 3, activation="relu"),  # (46, 46, 32)
        layers.MaxPooling2D(2),  # (23, 23, 32)
        layers.Conv2D(64, 3, activation="relu"),  # (21, 21, 64)
        layers.MaxPooling2D(2),  # (10, 10, 64)
        layers.Flatten(),  # 6400
        layers.Dense(NUM_CLASSES, activation="softmax"),
    ],
    name="cnn_base",
)

cnn.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
cnn.summary()

### Dimensions et paramètres couche par couche

Kernel 3x3, stride 1 et padding "valid" (les valeurs par défaut) : on n'ajoute pas de zéros autour, donc la sortie perd 2 pixels (48 - 3 + 1 = 46). Avec padding "same" on garderait 48. Le pooling 2x2 divise par 2 (21 est impair, ça donne 10, la dernière ligne et colonne sont perdues).

Paramètres d'une conv = (3 x 3 x nb de canaux en entrée + 1 biais) x nb de filtres. Les mêmes poids servent partout dans l'image, d'où beaucoup moins de paramètres que la baseline.

| Couche | Sortie | Paramètres |
|---|---|---:|
| Entrée | (48, 48, 1) | 0 |
| Conv2D(32, 3) + ReLU | (46, 46, 32) | (3x3x1 + 1) x 32 = 320 |
| MaxPooling2D(2) | (23, 23, 32) | 0 |
| Conv2D(64, 3) + ReLU | (21, 21, 64) | (3x3x32 + 1) x 64 = 18 496 |
| MaxPooling2D(2) | (10, 10, 64) | 0 |
| Flatten | (6400,) | 0 |
| Dense(7) + softmax | (7,) | 6400 x 7 + 7 = 44 807 |
| Total | | 63 623 |

Pourquoi cette architecture - c'est celle du sujet, simple à comprendre, et 2 blocs suffisent pour du 48x48 (on finit en 10x10). On double les filtres (32 puis 64) quand l'image rétrécit. Elle a 4,6 fois moins de paramètres que la baseline (63 623 contre 295 943). Les variantes (plus de couches, dropout...) sont testées en partie 6.

## Partie 4 - Entraînement du CNN

Nos choix :
- loss : categorical_crossentropy, parce qu'il y a une seule classe par image parmi 7, en one-hot, avec softmax en sortie. Elle devient très grande quand le modèle se trompe en étant sûr de lui.
- optimiseur : Adam avec le learning rate par défaut (0.001). Il marche bien sans réglage, et la val_loss baisse dès le début (voir 4b), donc pas de raison de le changer.
- batch size : 64, une valeur classique. Ça fait 359 mises à jour par époque et ça passe largement en mémoire.
- époques : 30 max avec early stopping (patience 3) sur val_loss. On ne sait pas à l'avance combien il en faut, c'est la validation qui décide.
- métriques : l'accuracy pendant l'entraînement, mais vu le déséquilibre on regarde aussi la matrice de confusion et le rappel par classe en partie 5 (un modèle peut avoir une accuracy correcte sans jamais trouver disgust).

On surveille val_loss plutôt que val_accuracy pour l'arrêt car elle bouge plus finement d'une époque à l'autre.

### 4a - Entraînement avec early stopping

Si val_loss ne s'améliore plus pendant 3 époques on arrête, et on reprend les poids de la meilleure époque (avant que le modèle commence à apprendre par coeur).

In [ ]:
EPOCHS_CNN = 30

arret_automatique = keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)

historique_cnn = cnn.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_CNN, callbacks=[arret_automatique])

perte_val_cnn, accuracy_val_cnn = cnn.evaluate(val_ds, verbose=0)
print("accuracy val CNN:", round(accuracy_val_cnn * 100, 1), "%")
print("accuracy val baseline:", round(accuracy_val * 100, 1), "%")

### 4b - Courbes du CNN

Même graphique qu'en 2c, avec en pointillés la meilleure époque (celle gardée par l'early stopping). On le met dans une fonction parce qu'on le réutilise en partie 6.

In [ ]:
def tracer_courbes(historique, titre):
    h = historique.history
    epoques = range(1, len(h["loss"]) + 1)
    meilleure = int(np.argmin(h["val_loss"])) + 1

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(epoques, h["loss"], label="train")
    ax1.plot(epoques, h["val_loss"], label="val")
    ax1.set_title("loss")
    ax2.plot(epoques, h["accuracy"], label="train")
    ax2.plot(epoques, h["val_accuracy"], label="val")
    ax2.set_title("accuracy")
    for ax in [ax1, ax2]:
        ax.axvline(meilleure, color="gray", linestyle="--", label="meilleure époque")
        ax.set_xlabel("époque")
        ax.legend()
    plt.suptitle(titre)
    plt.show()


tracer_courbes(historique_cnn, "CNN de base")

h = historique_cnn.history
m = int(np.argmin(h["val_loss"]))  # indice de la meilleure époque (0 = la 1ère)
print("époques faites:", len(h["loss"]), "/ meilleure:", m + 1)
print("meilleure époque: acc train", round(h["accuracy"][m] * 100, 1), "acc val", round(h["val_accuracy"][m] * 100, 1), "val_loss", round(h["val_loss"][m], 3))
print("dernière époque: acc train", round(h["accuracy"][-1] * 100, 1), "acc val", round(h["val_accuracy"][-1] * 100, 1), "val_loss", round(h["val_loss"][-1], 3))

Au début les deux loss baissent ensemble, puis elles se séparent : la loss train continue de baisser alors que la val_loss stagne puis remonte. Pour l'accuracy c'est pareil, celle du train continue de monter alors que celle de la val plafonne.

C'est du surapprentissage : à partir de la meilleure époque (la ligne en pointillés), le modèle commence à apprendre les images de train par coeur. L'early stopping s'arrête 3 époques plus tard et remet les poids de la meilleure époque. La cellule au-dessus affiche la meilleure époque et l'écart train/val de cette exécution. On essaie de réduire ce surapprentissage en partie 6 (dropout, augmentation).

## Partie 5 - Évaluation du CNN

Tout se fait sur la validation, le test est gardé pour la fin.

### 5a - Matrice de confusion et rappel par classe

Ligne = vraie classe, colonne = classe prédite, la diagonale = les bonnes réponses. Le rappel d'une classe c'est bonnes réponses / nombre d'images de la classe. Avec le déséquilibre c'est plus parlant que l'accuracy.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

# une seule boucle sur la val pour garder images, labels et prédictions dans le même ordre
images_lots = []
labels_lots = []
probabilites_lots = []
for images, labels in val_ds:
    images_lots.append(images.numpy())
    labels_lots.append(labels.numpy())
    probabilites_lots.append(cnn.predict_on_batch(images))
labels_onehot = np.concatenate(labels_lots)
probabilites_cnn = np.concatenate(probabilites_lots)

vraies_classes = labels_onehot.argmax(axis=1)
classes_predites = probabilites_cnn.argmax(axis=1)

matrice = confusion_matrix(vraies_classes, classes_predites)
ConfusionMatrixDisplay(matrice, display_labels=CLASS_NAMES).plot(cmap="Blues", xticks_rotation=45)
plt.title("matrice de confusion CNN (val)")
plt.show()

for i, classe in enumerate(CLASS_NAMES):
    bons = matrice[i, i]
    total = matrice[i].sum()
    print(classe, "rappel:", round(bons / total * 100, 1), "%", f"({bons}/{total})")

### 5b - Courbes ROC

Une courbe par classe (la classe contre toutes les autres). Plus l'AUC est proche de 1 mieux la classe est séparée, 0.5 c'est le hasard.

Attention, l'AUC regarde si les probas sont plus hautes pour les bonnes images, pas si la prédiction finale est juste. disgust peut avoir une bonne AUC et un mauvais rappel, donc on lit ça avec la matrice.

In [ ]:
from sklearn.metrics import auc, roc_curve

plt.figure(figsize=(7, 6))
for i, classe in enumerate(CLASS_NAMES):
    fpr, tpr, seuils = roc_curve(labels_onehot[:, i], probabilites_cnn[:, i])
    plt.plot(fpr, tpr, label=f"{classe} (AUC {auc(fpr, tpr):.2f})")
plt.plot([0, 1], [0, 1], "k--", label="hasard")
plt.xlabel("taux de faux positifs")
plt.ylabel("taux de vrais positifs")
plt.title("ROC par classe (val)")
plt.legend()
plt.show()

### 5c - Classes les plus confondues et les plus difficiles

On prend toutes les cases hors diagonale de la matrice et on les trie. On donne aussi le % par rapport à la vraie classe, car les classes n'ont pas la même taille (50 erreurs c'est énorme pour disgust qui a 70 images en val, pas pour happy).

In [ ]:
confusions = []
for vraie in range(NUM_CLASSES):
    for predite in range(NUM_CLASSES):
        if vraie != predite:
            confusions.append((matrice[vraie, predite], vraie, predite))
confusions.sort(reverse=True)

print("les 5 confusions les plus fréquentes (vraie / prédite):")
for nombre, vraie, predite in confusions[:5]:
    pourcent = nombre / matrice[vraie].sum() * 100
    print(CLASS_NAMES[vraie], "/", CLASS_NAMES[predite], ":", nombre, "images", f"({pourcent:.1f}% des {CLASS_NAMES[vraie]})")

print()
print("classes de la plus dure à la plus facile:")
rappels = matrice.diagonal() / matrice.sum(axis=1)
for i in np.argsort(rappels):
    print(CLASS_NAMES[i], round(rappels[i] * 100, 1), "%")

happy et surprise sont les mieux reconnues : le sourire pour l'une, la bouche ouverte et les yeux grands ouverts pour l'autre, ça se voit même en 48x48. happy a aussi le plus d'images d'entraînement.

Les plus grosses confusions passent presque toutes par sad et neutral : neutral prise pour sad, fear prise pour sad, sad prise pour neutral, angry prise pour sad. Un visage triste, neutre ou inquiet a souvent la bouche fermée, la différence se joue à quelques pixels autour des sourcils et des coins de la bouche.

Les classes les plus dures sont disgust, fear et angry :
- disgust : surtout un problème de quantité, environ 16 fois moins d'images en train que happy. Le modèle la voit rarement et se tromper dessus coûte peu à l'accuracy.
- fear : elle n'a pas de signe à elle. Elle est prise pour sad, angry, neutral ou surprise (yeux et bouche ouverts).
- angry : souvent prise pour sad ou neutral, sans doute parce que des sourcils froncés se voient mal à cette résolution.

Autre cause probable, qu'on n'a pas mesurée : des étiquettes fausses dans FER2013. Les images viennent du web et certaines expressions sont ambiguës même pour nous (voir 5d).

Les nombres exacts sont affichés au-dessus. Ils bougent un peu d'une exécution à l'autre, mais ces tendances restent les mêmes.

### 5d - Exemples : image, vraie classe, prédiction, proba

4 images bien classées (en vert) et les 8 erreurs où le modèle était le plus sûr de lui (en rouge), ce sont les plus intéressantes à regarder.

In [ ]:
images_val_toutes = np.concatenate(images_lots)

bonnes = np.where(classes_predites == vraies_classes)[0]
fausses = np.where(classes_predites != vraies_classes)[0]

# on trie les erreurs par proba de la classe prédite, de la plus haute à la plus basse
confiance = probabilites_cnn[fausses].max(axis=1)
ordre = np.argsort(-confiance)
fausses_choisies = fausses[ordre[:8]]
exemples = list(bonnes[:4]) + list(fausses_choisies)

fig, axes = plt.subplots(2, 6, figsize=(14, 6))
for ax, n in zip(axes.flat, exemples):
    vraie = vraies_classes[n]
    predite = classes_predites[n]
    if vraie == predite:
        couleur = "green"
    else:
        couleur = "red"
    ax.imshow(images_val_toutes[n], cmap="gray", vmin=0, vmax=1)
    titre = f"vraie: {CLASS_NAMES[vraie]} ({probabilites_cnn[n, vraie]:.0%})\npredite: {CLASS_NAMES[predite]} ({probabilites_cnn[n, predite]:.0%})"
    ax.set_title(titre, color=couleur, fontsize=10)
    ax.axis("off")
plt.suptitle("4 bonnes prédictions et 8 erreurs (val)")
plt.tight_layout()
plt.show()

Sur les erreurs affichées, le modèle met presque toute la probabilité sur la mauvaise classe et presque rien sur la vraie. Les images changent un peu d'une exécution à l'autre, mais on retrouve à chaque fois les mêmes types d'erreurs :
- des expressions qui se ressemblent : fear prise pour surprise (yeux écarquillés, bouche ouverte), fear ou angry prise pour une autre classe quand la personne crie, neutral prise pour surprise quand la bouche est ouverte (bâillement) ;
- des étiquettes qui ont l'air fausses : une personne qui sourit étiquetée neutral ou surprise et prédite happy, la prédiction a l'air plus juste que le label ;
- des images difficiles : visage tout petit dans l'image, tête penchée, yeux plissés.

Le modèle a l'air de se baser surtout sur la bouche et l'ouverture des yeux, moins sur les sourcils. Et les étiquettes douteuses limitent sûrement le score max qu'on peut avoir sur FER2013.

## Partie 6 - Expériences

La référence c'est le CNN de la partie 4 (cnn_base). Chaque expérience part du dernier modèle gardé et change une seule chose, pour savoir d'où vient l'effet. Le reste ne bouge pas : même découpage, seed remise à 42 avant chaque modèle, Adam lr 0.001, batch 64, 30 époques max avec early stopping (patience 3). Tout est mesuré sur la validation.

Pour chaque modèle on note l'accuracy et la loss de val, la meilleure époque, l'écart d'accuracy train - val (pour le surapprentissage) et le rappel de disgust (la classe la plus rare).

| | Changement | Pourquoi on essaie |
|---|---|---|
| E1 | une couche Dense(128) + ReLU avant la sortie (comme la figure 2 du sujet) | combiner les motifs trouvés par les convolutions avant de décider |
| E2 | Dropout(0.5) avant la Dense(128) | le modèle surapprend vite (4b) |
| E3 | 3e bloc Conv2D(128) + MaxPooling | motifs plus grands, et moins de valeurs en entrée de la Dense |
| E4 | data augmentation (miroir, petite rotation, petit zoom) | le modèle voit des variantes à chaque époque, il apprend moins par coeur |
| E5 | class_weight | une erreur sur disgust coûte plus cher, son rappel devrait monter |

Règle fixée avant de voir les résultats : on garde un changement s'il améliore l'accuracy de val. Si l'écart est de moins d'un point, on regarde la val_loss et l'écart train/val.

### 6a - Fonctions pour les expériences

`entrainer` compile et entraîne un modèle comme en partie 4. `mesurer` calcule les scores sur la validation et ajoute une ligne dans `resultats` (pour le tableau de la fin). On commence par mesurer cnn_base.

In [ ]:
resultats = []
NUMERO_DISGUST = CLASS_NAMES.index("disgust")


def entrainer(modele, poids_classes=None):
    modele.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
    return modele.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS_CNN,
        callbacks=[arret_automatique],
        class_weight=poids_classes,
        verbose=2,  # 1 ligne par époque
    )


def mesurer(nom, changement, modele, historique):
    perte, accuracy = modele.evaluate(val_ds, verbose=0)

    # val_ds est en cache donc toujours dans le même ordre que vraies_classes (5a)
    predites = modele.predict(val_ds, verbose=0).argmax(axis=1)
    est_disgust = vraies_classes == NUMERO_DISGUST
    rappel_disgust = (predites[est_disgust] == NUMERO_DISGUST).mean()

    h = historique.history
    m = int(np.argmin(h["val_loss"]))
    ecart = h["accuracy"][m] - h["val_accuracy"][m]

    resultats.append([nom, changement, accuracy, perte, m + 1, len(h["loss"]), ecart, rappel_disgust])
    print(nom, "| acc", round(accuracy * 100, 1), "| loss", round(perte, 3), "| meilleure ép.", m + 1, "/", len(h["loss"]),
          "| écart", round(ecart * 100, 1), "| rappel disgust", round(rappel_disgust * 100, 1))


mesurer("cnn_base", "référence (partie 3)", cnn, historique_cnn)

### 6b - E1 : une couche Dense(128) cachée

Dans cnn_base les 6400 valeurs vont directement aux 7 sorties. Avec une couche de 128 neurones entre les deux, le modèle peut combiner les motifs avant de décider (par exemple coins de la bouche relevés + yeux plissés). C'est aussi le schéma de la figure 2 du sujet.

Le nombre de paramètres passe de 63 623 à 839 047, presque tout dans la nouvelle couche (6400 x 128 + 128 = 819 328).

In [ ]:
keras.utils.set_random_seed(SEED)

modele_e1 = keras.Sequential(
    [
        keras.Input(shape=IMAGE_SIZE + (1,)),
        layers.Conv2D(32, 3, activation="relu"),
        layers.MaxPooling2D(2),
        layers.Conv2D(64, 3, activation="relu"),
        layers.MaxPooling2D(2),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),  # nouveau
        layers.Dense(NUM_CLASSES, activation="softmax"),
    ],
    name="e1_dense128",
)
historique_e1 = entrainer(modele_e1)
mesurer("e1_dense128", "+ Dense(128)", modele_e1, historique_e1)

Le gain d'accuracy est petit, de l'ordre d'un point, donc du même ordre que ce qui change d'une exécution à l'autre. La val_loss baisse un peu. Par contre il surapprend plus tôt : sa meilleure époque arrive avant celle de cnn_base et l'écart train/val augmente. Logique avec 13 fois plus de paramètres pour les mêmes images.

On garde : la loss est meilleure et c'est le schéma de la figure 2 du sujet. Son surapprentissage c'est justement ce qu'on teste avec E2.

### 6c - E2 : Dropout

Avec E1 la meilleure époque arrive tôt, et presque tous les paramètres sont dans la Dense(128) (819 328 sur 839 047). On ajoute un Dropout(0.5) juste avant. Pendant l'entraînement il met à 0 la moitié des valeurs au hasard, différentes à chaque lot, donc le réseau ne peut pas compter sur quelques valeurs précises. En prédiction il est désactivé. Il n'ajoute pas de paramètre.

In [ ]:
keras.utils.set_random_seed(SEED)

modele_e2 = keras.Sequential(
    [
        keras.Input(shape=IMAGE_SIZE + (1,)),
        layers.Conv2D(32, 3, activation="relu"),
        layers.MaxPooling2D(2),
        layers.Conv2D(64, 3, activation="relu"),
        layers.MaxPooling2D(2),
        layers.Flatten(),
        layers.Dropout(0.5),  # nouveau, actif seulement pendant l'entraînement
        layers.Dense(128, activation="relu"),
        layers.Dense(NUM_CLASSES, activation="softmax"),
    ],
    name="e2_dropout",
)
historique_e2 = entrainer(modele_e2)
mesurer("e2_dropout", "+ Dropout(0.5)", modele_e2, historique_e2)

L'accuracy bouge de moins d'un point par rapport à E1, donc d'après notre règle on regarde la loss et l'écart train/val. Les deux sont meilleurs : la val_loss baisse et l'écart train/val est nettement plus petit, le dropout réduit bien le surapprentissage. On garde.

### 6d - E3 : un 3e bloc de convolution

Avec 2 blocs, chaque valeur de la dernière couche ne voit qu'un bout du visage. Un 3e bloc combine ces bouts en motifs plus grands (la bouche avec les joues, les sourcils avec les yeux). Il réduit aussi la taille avant Flatten : 2048 valeurs au lieu de 6400 en entrée de la Dense, donc moins de poids.

On ajoute Conv2D(128, 3) + MaxPooling2D(2) en doublant encore les filtres (32, 64, 128).

| Couche | Sortie | Paramètres |
|---|---|---:|
| après le 2e bloc | (10, 10, 64) | |
| Conv2D(128, 3) + ReLU | (8, 8, 128) | (3x3x64 + 1) x 128 = 73 856 |
| MaxPooling2D(2) | (4, 4, 128) | 0 |
| Flatten | (2048,) | 0 |
| Dense(128) | (128,) | 2048 x 128 + 128 = 262 272 |
| Total du modèle | | 355 847 (839 047 pour E2) |

In [ ]:
keras.utils.set_random_seed(SEED)

modele_e3 = keras.Sequential(
    [
        keras.Input(shape=IMAGE_SIZE + (1,)),
        layers.Conv2D(32, 3, activation="relu"),
        layers.MaxPooling2D(2),
        layers.Conv2D(64, 3, activation="relu"),
        layers.MaxPooling2D(2),
        layers.Conv2D(128, 3, activation="relu"),  # nouveau 3e bloc : (8, 8, 128)
        layers.MaxPooling2D(2),  # (4, 4, 128)
        layers.Flatten(),  # 2048
        layers.Dropout(0.5),
        layers.Dense(128, activation="relu"),
        layers.Dense(NUM_CLASSES, activation="softmax"),
    ],
    name="e3_3_blocs",
)
historique_e3 = entrainer(modele_e3)
mesurer("e3_3_blocs", "+ 3e bloc Conv2D(128)", modele_e3, historique_e3)

C'est le plus gros gain de toutes les expériences : plusieurs points d'accuracy en plus et la val_loss la plus basse, avec moins de la moitié des paramètres de E2. disgust est aussi mieux reconnu et l'écart train/val reste petit. La meilleure époque arrive plus tard, le modèle continue de progresser plus longtemps avant de surapprendre.

On garde. Ici ajouter des convolutions a mieux marché que la grosse couche dense de E1.

### 6e - E4 : data augmentation

E3 surapprend encore un peu (sa val_loss remonte après sa meilleure époque). Avec l'augmentation, le modèle ne voit jamais 2 fois exactement la même image. On ajoute 3 couches au début du modèle :
- RandomFlip("horizontal") : miroir gauche/droite, une expression reste la même dans un miroir (pas de haut/bas, il n'y a pas de visages à l'envers)
- RandomRotation(0.05) : rotation de 18° max (0.05 tour), comme une tête un peu penchée
- RandomZoom(0.1) : zoom avant ou arrière de 10% max

Ces couches ne marchent que pendant l'entraînement. En val, en test et pour la démo l'image n'est pas modifiée.

In [ ]:
keras.utils.set_random_seed(SEED)

modele_e4 = keras.Sequential(
    [
        keras.Input(shape=IMAGE_SIZE + (1,)),
        # nouveau : augmentation, seulement pendant l'entraînement
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.05),
        layers.RandomZoom(0.1),
        layers.Conv2D(32, 3, activation="relu"),
        layers.MaxPooling2D(2),
        layers.Conv2D(64, 3, activation="relu"),
        layers.MaxPooling2D(2),
        layers.Conv2D(128, 3, activation="relu"),
        layers.MaxPooling2D(2),
        layers.Flatten(),
        layers.Dropout(0.5),
        layers.Dense(128, activation="relu"),
        layers.Dense(NUM_CLASSES, activation="softmax"),
    ],
    name="e4_augmentation",
)
historique_e4 = entrainer(modele_e4)
mesurer("e4_augmentation", "+ augmentation", modele_e4, historique_e4)

L'écart train/val devient presque nul, voire négatif : il ne surapprend plus. Mais il apprend plus lentement, sa meilleure époque arrive bien plus tard que pour E3. Avec le même protocole que les autres (30 époques max, patience 3), son accuracy de val reste en dessous de E3 et disgust est moins bien reconnu. Avec plus d'époques ou une patience plus grande ça ferait peut-être mieux, mais on ne l'a pas testé (ce serait une autre expérience).

On ne garde pas. E5 repart donc de E3.

### 6f - E5 : poids par classe (class_weight)

Même avec E3, disgust reste la classe la moins bien trouvée. Le modèle la voit 16 fois moins souvent que happy, donc ses erreurs dessus comptent peu dans la loss. Avec class_weight, l'erreur sur une image est multipliée par le poids de sa classe : poids = nb total d'images / (7 x nb d'images de la classe). Une classe moyenne a un poids autour de 1, une classe rare un poids plus grand.

Même modèle que E3 (E4 n'a pas été gardé). Les poids sont calculés avec les comptes de 1a (tout le dossier train). La validation est tirée au hasard dedans, donc les proportions sont presque les mêmes.

In [ ]:
total_images = sum(nombres_train)
poids_classes = {}
for i, nombre in enumerate(nombres_train):
    poids_classes[i] = total_images / (NUM_CLASSES * nombre)
    print(CLASS_NAMES[i], nombre, "images, poids", round(poids_classes[i], 2))

keras.utils.set_random_seed(SEED)
modele_e5 = keras.Sequential(
    [
        keras.Input(shape=IMAGE_SIZE + (1,)),
        layers.Conv2D(32, 3, activation="relu"),
        layers.MaxPooling2D(2),
        layers.Conv2D(64, 3, activation="relu"),
        layers.MaxPooling2D(2),
        layers.Conv2D(128, 3, activation="relu"),
        layers.MaxPooling2D(2),
        layers.Flatten(),
        layers.Dropout(0.5),
        layers.Dense(128, activation="relu"),
        layers.Dense(NUM_CLASSES, activation="softmax"),
    ],
    name="e5_poids_classes",
)
historique_e5 = entrainer(modele_e5, poids_classes)  # nouveau : poids par classe
mesurer("e5_poids_classes", "+ class_weight", modele_e5, historique_e5)

# rappel de chaque classe avant (E3) et après (E5)
predites_e3 = modele_e3.predict(val_ds, verbose=0).argmax(axis=1)
predites_e5 = modele_e5.predict(val_ds, verbose=0).argmax(axis=1)
for i, classe in enumerate(CLASS_NAMES):
    est_classe = vraies_classes == i
    r3 = (predites_e3[est_classe] == i).mean()
    r5 = (predites_e5[est_classe] == i).mean()
    print(classe, "E3:", round(r3 * 100, 1), "E5:", round(r5 * 100, 1))

Le rappel de disgust monte beaucoup (la cellule affiche le rappel de chaque classe avec E3 et avec E5), donc les poids font ce qu'on attendait. Mais l'accuracy baisse de plusieurs points et la val_loss monte. happy perd le plus, neutral, sad et angry un peu : le modèle répond plus souvent les classes rares, et une partie de ces réponses tombe sur des images d'autres classes. disgust c'est seulement 70 images sur 5 739 en val, gagner sur cette classe ne compense pas ce qu'on perd ailleurs.

On ne garde pas. Le poids de disgust (environ 9) est sûrement trop fort, un poids plus doux (par exemple la racine carrée) serait à tester, on ne l'a pas fait.

### 6g - Tableau comparatif

écart = accuracy train - accuracy val à la meilleure époque. Avec dropout ou augmentation l'accuracy train est calculée sur des images modifiées, donc l'écart paraît un peu plus petit qu'en vrai.

In [ ]:
print("modèle | changement | acc val | loss val | meilleure ép. | écart | rappel disgust")
for r in resultats:
    nom, changement, acc, perte, meilleure, nb_epoques, ecart, rappel_d = r
    print(nom, "|", changement, "|", round(acc * 100, 1), "|", round(perte, 3), "|", meilleure, "/", nb_epoques,
          "|", round(ecart * 100, 1), "|", round(rappel_d * 100, 1))

Résultats de notre exécution de référence, le 8 octobre 2026 (notebook complet en local sur CPU, Windows, TensorFlow 2.21, seed 42). Si on relance, surtout sur GPU, les chiffres affichés au-dessus peuvent bouger un peu.

| Modèle | Changement | Acc val | Loss val | Meilleure ép. | Écart train-val | Rappel disgust | Observation | Décision |
|---|---|---:|---:|---:|---:|---:|---|---|
| cnn_base | référence (partie 3) | 50.8% | 1.339 | 8/11 | +6.2 | 11.4% | surapprend après quelques époques | référence |
| E1 | + Dense(128) | 51.9% | 1.303 | 6/9 | +7.9 | 12.9% | petit gain, surapprend plus tôt | gardé |
| E2 | + Dropout(0.5) | 52.2% | 1.274 | 7/10 | +3.6 | 18.6% | loss plus basse, écart train/val divisé par 2 | gardé (loss et écart meilleurs) |
| E3 | + 3e bloc Conv2D(128) | 58.3% | 1.127 | 14/17 | +3.4 | 25.7% | plus gros gain, 2,4 fois moins de paramètres que E2 | gardé, modèle final |
| E4 | + augmentation | 56.6% | 1.144 | 20/23 | -3.0 | 10.0% | ne surapprend plus mais apprend plus lentement | pas gardé |
| E5 | E3 + class_weight | 53.8% | 1.238 | 15/18 | +1.7 | 60.0% | disgust bien mieux trouvé mais happy baisse beaucoup | pas gardé |

Pour comparer : baseline dense 34.0%, hasard 14.3%, toujours happy 24.4%.

Modèle final : E3. Il a la meilleure accuracy et la plus petite loss de val, c'est la règle qu'on avait fixée au début. Par rapport au CNN de départ il gagne plusieurs points et trouve plus d'images disgust. E5 trouve beaucoup mieux disgust mais perd en accuracy, on ne le prend pas car on compare d'abord sur l'accuracy. Si le but était de détecter le dégoût à tout prix, on prendrait E5.

Les écarts de moins d'un point (cnn_base, E1, E2) peuvent venir du hasard. Les effets de E3, E4 et E5 sont assez grands pour se retrouver à chaque exécution.

### 6h - Courbes du modèle final

Pour comparer avec celles du CNN de base (4b).

In [ ]:
modele_final = modele_e3
tracer_courbes(historique_e3, "modèle final (E3)")

Par rapport à cnn_base (4b), les courbes train et val restent ensemble plus longtemps et la val_loss descend plus bas avant de remonter. Le surapprentissage arrive plus tard et plus doucement, mais il est toujours là : à la fin l'accuracy train est au-dessus de celle de la val.

### 6i - Évaluation finale sur le test

C'est la seule fois qu'on utilise le test. Tous les choix ont été faits avant, sur la validation. Si on utilisait le test pour choisir le modèle, son score ne voudrait plus rien dire sur des images jamais vues, donc on ne revient pas modifier le modèle après cette cellule.

On sauvegarde aussi le modèle dans `models/modele_final.keras` pour la démo.

In [ ]:
perte_test, accuracy_test = modele_final.evaluate(test_ds, verbose=0)

# test_ds n'est pas mélangé (shuffle=False) donc labels et prédictions sont dans le même ordre
labels_test = []
for images, labels in test_ds:
    labels_test.append(labels.numpy())
vraies_test = np.concatenate(labels_test).argmax(axis=1)
predites_test = modele_final.predict(test_ds, verbose=0).argmax(axis=1)

matrice_test = confusion_matrix(vraies_test, predites_test)
ConfusionMatrixDisplay(matrice_test, display_labels=CLASS_NAMES).plot(cmap="Blues", xticks_rotation=45)
plt.title("modèle final, matrice de confusion (test)")
plt.show()

perte_val_final, accuracy_val_final = modele_final.evaluate(val_ds, verbose=0)
nb_happy_test = (vraies_test == CLASS_NAMES.index("happy")).sum()

print("accuracy test:", round(accuracy_test * 100, 1), "% / loss", round(perte_test, 3), "/", len(vraies_test), "images")
print("accuracy val du même modèle:", round(accuracy_val_final * 100, 1), "%")
print("toujours happy sur le test:", round(nb_happy_test / len(vraies_test) * 100, 1), "%")
for i, classe in enumerate(CLASS_NAMES):
    bons = matrice_test[i, i]
    total = matrice_test[i].sum()
    print(classe, "rappel:", round(bons / total * 100, 1), "%", f"({bons}/{total})")

Path("models").mkdir(exist_ok=True)
modele_final.save("models/modele_final.keras")
print("modèle sauvegardé")

Le score sur le test est très proche de celui de la validation (les deux sont affichés au-dessus). Donc le choix fait sur la validation tient sur des images qui n'avaient servi à rien avant. Il est bien au-dessus d'un modèle qui répondrait toujours happy.

L'ordre des classes est presque le même qu'en val : happy et surprise en tête, puis neutral, angry et sad, et disgust et fear restent les plus dures. Les grosses confusions sont aussi les mêmes, presque toutes autour de sad et neutral, et disgust est surtout pris pour angry.

Résultat de notre exécution de référence (8 octobre 2026, CPU, Windows, TensorFlow 2.21) : 57.7% sur les 7 177 images de test (loss 1.124), contre 58.3% en validation. Toujours happy donnerait 24.7%.

| Classe | angry | disgust | fear | happy | neutral | sad | surprise |
|---|---:|---:|---:|---:|---:|---:|---:|
| Rappel (test) | 50.1% | 35.1% | 27.5% | 81.8% | 57.9% | 46.0% | 72.4% |

### Démo

On recharge le modèle sauvegardé et on prédit une image. Pour en tester une autre il suffit de changer `CHEMIN_IMAGE`.

L'image doit être préparée comme à l'entraînement (gris, 48x48, divisée par 255), sinon les prédictions n'ont pas de sens. Le modèle a appris sur des visages déjà recadrés, donc sur une photo où le visage est petit ou pas centré il va se tromper (c'est le rôle de la détection de visages des parties 8 et 9).

In [ ]:
CHEMIN_IMAGE = sorted((DATA_DIR / "test" / "happy").glob("*.jpg"))[0]  # changer ici pour tester une autre image

modele_demo = keras.models.load_model("models/modele_final.keras")

# même préparation qu'à l'entraînement
image = keras.utils.load_img(CHEMIN_IMAGE, color_mode="grayscale", target_size=IMAGE_SIZE)
pixels = keras.utils.img_to_array(image) / 255.0  # (48, 48, 1)
lot = np.expand_dims(pixels, axis=0)  # (1, 48, 48, 1), predict veut un lot
probabilites_demo = modele_demo.predict(lot, verbose=0)[0]
classe_demo = CLASS_NAMES[int(probabilites_demo.argmax())]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))
ax1.imshow(pixels, cmap="gray", vmin=0, vmax=1)
ax1.set_title(f"prédiction: {classe_demo} ({probabilites_demo.max():.0%})")
ax1.axis("off")
ax2.barh(CLASS_NAMES, probabilites_demo)
ax2.invert_yaxis()
ax2.set_xlim(0, 1)
ax2.set_xlabel("probabilité")
plt.tight_layout()
plt.show()

print("image:", CHEMIN_IMAGE)
print("vraie classe (dossier):", Path(CHEMIN_IMAGE).parent.name)
for classe, p in zip(CLASS_NAMES, probabilites_demo):
    print(classe, round(float(p), 2))